# Dataset Viewer (Kaggle version)

Drag the slider to flip through any dataset, one image at a time.

Works with MNIST, FashionMNIST, CIFAR-10, KITTI, COCO, INRIA Person — or any torchvision-style dataset where indexing returns an `(image, label)` pair.

## Step 1 — Load a dataset

On Kaggle, attach KITTI and COCO with **Add Input** in the right-hand panel (no download): search `klemenko/kitti-dataset` or `awsaf49/coco-2017-dataset`. MNIST, FashionMNIST, CIFAR-10 and INRIA still download, so switch **Internet** on in the notebook settings.

Uncomment exactly one of the lines below. The rest of the notebook only needs a variable called `dataset`.

In [ ]:
import glob, os
from torchvision import datasets, transforms

transform = transforms.ToTensor()

def find(path):
    # Kaggle mounts attached datasets read-only under /kaggle/input; how deep a folder sits varies by upload
    hits = glob.glob(f'/kaggle/input/**/{path}', recursive=True)
    assert hits, f'{path} not found under /kaggle/input: attach the dataset with Add Input first'
    return hits[0]

def kitti_root():
    # torchvision expects <root>/Kitti/raw/training/{image_2,label_2}; link the read-only Kaggle folders into that shape
    for d in ('image_2', 'label_2'):
        link = f'data/Kitti/raw/training/{d}'
        os.makedirs(os.path.dirname(link), exist_ok=True)
        if not os.path.islink(link):
            os.symlink(find(f'training/{d}'), link)
    return 'data'

# --- Pick ONE: ---
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
# dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
# dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
# dataset = datasets.Kitti(root=kitti_root(), train=True, transform=transform)  # attach klemenko/kitti-dataset
# dataset = datasets.CocoDetection(find('val2017'), find('annotations/instances_val2017.json'), transform=transform)  # attach awsaf49/coco-2017-dataset
# dataset = datasets.ImageFolder('./data/INRIAPerson/Train', transform=transform, allow_empty=True)  # see Tips

print(f'Loaded {len(dataset)} images')
print(f'Classes: {getattr(dataset, "classes", "boxes per image (detection)")}')

## Step 2 — Browse the images

Run the cell below. A control panel will appear with:

- **⏮ First** and **Last ⏭** — jump to the very first / very last image
- **← Previous** and **Next →** — step one image at a time
- **Random** — pick a random image (great for skimming the dataset)
- **Go to:** — type any image number into the box, then **click the blue Go button** to jump there

A counter shows how many images are in the dataset and which one you're looking at.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

total = len(dataset)
current_idx = 0

# --- Build the controls ---
first_btn  = widgets.Button(description='⏮ First',     layout=widgets.Layout(width='100px'))
prev_btn   = widgets.Button(description='← Previous',  layout=widgets.Layout(width='120px'))
next_btn   = widgets.Button(description='Next →',      layout=widgets.Layout(width='120px'))
last_btn   = widgets.Button(description='Last ⏭',      layout=widgets.Layout(width='100px'))
random_btn = widgets.Button(description='Random',      layout=widgets.Layout(width='100px'))

idx_input  = widgets.IntText(value=0, description='Go to:',
                             layout=widgets.Layout(width='180px'))
go_btn     = widgets.Button(description='Go', button_style='primary',
                            layout=widgets.Layout(width='60px'))

counter    = widgets.HTML(value='')
picture    = widgets.Output()

def render(idx):
    global current_idx
    # Clamp to a valid range so a bad input can't crash anything
    idx = max(0, min(int(idx), total - 1))
    current_idx = idx

    counter.value = (
        f"<b>Image {idx + 1:,} of {total:,}</b> "
        f"<span style='color:#888'>(index {idx})</span>"
    )

    image, label = dataset[idx]
    if isinstance(image, torch.Tensor):
        image = image.numpy()
        if image.ndim == 3:
            image = image.transpose(1, 2, 0)
    image = np.squeeze(image)
    cmap = 'gray' if image.ndim == 2 else None

    with picture:
        clear_output(wait=True)
        plt.close('all')                    # avoid stacking old figures
        h, w = image.shape[:2]
        fig, ax = plt.subplots(figsize=(min(4 * w / h, 12), 4))
        ax.imshow(image, cmap=cmap)
        if isinstance(label, list):         # detection (KITTI, COCO): a list of boxes
            for obj in label:
                if 'type' in obj:           # KITTI box: [x1, y1, x2, y2]
                    x1, y1, x2, y2 = obj['bbox']
                    name = obj['type']
                else:                       # COCO box: [x, y, width, height]
                    x1, y1, bw, bh = obj['bbox']
                    x2, y2 = x1 + bw, y1 + bh
                    name = dataset.coco.cats[obj['category_id']]['name']
                ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='lime'))
                ax.text(x1, y1, name, color='lime', fontsize=8, va='bottom')
            ax.set_title(f'{len(label)} objects')
        elif label is None:                 # KITTI test split has no labels
            ax.set_title('No labels')
        else:
            class_name = dataset.classes[label] if hasattr(dataset, 'classes') else label
            ax.set_title(f'Label: {label}  ({class_name})')
        ax.axis('off')
        plt.show()

# --- Wire up the buttons ---
def on_first(b):  render(0)
def on_prev(b):   render((current_idx - 1) % total)
def on_next(b):   render((current_idx + 1) % total)
def on_last(b):   render(total - 1)
def on_random(b): render(int(np.random.randint(0, total)))
def on_go(b):     render(idx_input.value)

first_btn.on_click(on_first)
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
last_btn.on_click(on_last)
random_btn.on_click(on_random)
go_btn.on_click(on_go)

# --- Lay it out and show ---
row1 = widgets.HBox([first_btn, prev_btn, next_btn, last_btn, random_btn])
row2 = widgets.HBox([idx_input, go_btn, counter])
display(widgets.VBox([row1, row2, picture]))
render(0)

### Tips

- **Switch dataset:** go back to Step 1, comment out the current line, uncomment another, run it again, then re-run Step 2.
- **KITTI:** each image shows its labelled boxes (cars, pedestrians, cyclists…). `train=False` gives the test split, which has no labels.
- **COCO outside Kaggle:** not downloaded by torchvision. Run these once in a cell (about 1.07 GB: the 5,000 val2017 images plus annotations), then use the `CocoDetection` line:
  ```
  !pip install pycocotools
  !mkdir -p data && cd data && wget -q http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip
  !cd data && wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q annotations_trainval2017.zip
  ```
- **INRIA Person:** not built into torchvision, and the original INRIA site is often down. The Kaggle copies don't fit this viewer (no images without people), so download it here. Run this once in a cell (about 1 GB, a copy hosted by OpenIMAJ), then use the `ImageFolder` line:
  ```
  !mkdir -p data && cd data && wget -q http://datasets.openimaj.org/INRIAPerson.zip && unzip -q INRIAPerson.zip
  ```
  `ImageFolder` reads `Train/pos` (614 photos with people) and `Train/neg` (1,218 without) as classes. The `annotations` folder shows up as a third, empty class; ignore it. Swap `Train` for `Test` to browse the test split.
- **Your own dataset:** as long as `dataset[i]` returns `(image, label)` and `len(dataset)` works, the viewer will handle it — grayscale or colour, tensors or PIL images.